# Putting Data into Parquet

In [1]:
import json
import pandas as pd

In [33]:
DATA_PATH = "../Data/idiom_repository_all.json"

In [ ]:
with open(DATA_PATH, "rt", encoding="utf8") as f:
    data = json.load(f)

df_raw = []

for i in range(len(data)):
    tmp = {}
    idiom = data[i]

    idiom_keys = idiom.keys()
    
    for ikey in idiom_keys:
        d1 = idiom[ikey]

        if (isinstance(d1, list)):
            # Lists can have a dictionary inside of them
            if (len(d1) > 0 and isinstance(d1[0], dict)):
                d2 = d1[0]
                # There could be more than one dictionary...
                for ekey in d2.keys():
                    if (isinstance(d2[ekey], list)):
                        tmp[ekey] = d2[ekey][0]
                    else:
                        tmp[ekey] = d2[ekey]

            elif (len(d1) > 0 and isinstance(d1[0], list)):
                print(ikey, d1[0])

            else:
                tmp[ikey] = d1

        elif (isinstance(d1, str)):
            tmp[ikey] = d1
        elif (isinstance(d1, int)):
            tmp[ikey] = d1

    df_raw.append(tmp)

df = pd.DataFrame(df_raw)
df.to_parquet("../Data/idiom_repository_all.parquet")

# Something

In [1]:
import duckdb

In [2]:
DATA_PATH = "../Data/idiom_repository_all.parquet"

In [9]:
duckdb.query(f"""SELECT * FROM '{DATA_PATH}'""")

┌────────────┬─────────────────────────────────┬─────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬───────┬────────────┬───────┐
│ variations │              idiom              │         sources         │                                                                                                                                                           usages                                                                                                                                                            │                                           

In [3]:
query = f"""
    (SELECT 
        TRIM(variations, '[""]') AS all_variations
    FROM
        '{DATA_PATH}'
    WHERE
        len(variations) > 2)
    UNION
    (SELECT
        idiom
    FROM
        '{DATA_PATH}'
    )
"""
df = duckdb.query(query).df()

In [4]:
import spacy
from spacy.matcher import PhraseMatcher

In [5]:
nlp = spacy.load("en_core_web_sm")

In [6]:
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
patterns = [nlp.make_doc(p) for p in df["all_variations"].values]

matcher.add("PHRASES", patterns)

In [7]:
doc = nlp(
"""
We tried to keep the party a surprise, but Sam let the cat out of the bag.
I can kill two birds with one stone by stopping at the post office on my way to work.
That guy was so gone, he was two sheets to the wind. 
The bar was so crazy it was a free-for-all. 
Make 'er indoors get us a couple of sandwiches.
""")

In [17]:
matches = matcher(doc)
print(f"{"Match ID":20s} | {"Text":30s} | {"Start":5s} | {"End":5s}")
print(f"--------------------------------------------------------------------")
for match_id, start, end in matches:
    span = doc[start:end]
    print(f"{str(match_id):20s} | {span.text:30s} | {str(start):5} | {str(end):5s}")

Match ID             | Text                           | Start | End  
--------------------------------------------------------------------
1303609015319360366  | let the cat out of the bag     | 12    | 19   
1303609015319360366  | kill two birds with one stone  | 23    | 29   
1303609015319360366  | two birds with one stone       | 24    | 29   
1303609015319360366  | free-for-all                   | 65    | 70   
1303609015319360366  | 'er indoors                    | 73    | 76   
